# CQTNet: Cover Song Identification on Indian Music

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
!pip install -q torch torchvision torchaudio
!pip install -q librosa numpy matplotlib scikit-learn soundfile

## Imports

In [22]:
import os
import glob
import random
import warnings

import numpy as np
import librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import OrderedDict, defaultdict

warnings.filterwarnings('ignore')

## Config

In [23]:
DATA_DIR = '/content/drive/MyDrive/dataset/'
CQT_DIR = '/content/cqt_features/'
NUM_CLASSES = 245
BATCH_SIZE = 32
NUM_EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
OUT_LENGTH = 200
SEED = 42

## Setup

In [24]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## CQT Feature Extraction

In [25]:
def compute_cqt_features(audio_path, sr=22050, mean_size=20):
    """Compute temporally-pooled CQT magnitude spectrogram."""
    data, sr = librosa.load(audio_path, sr=sr)
    cqt = np.abs(librosa.cqt(y=data, sr=sr))
    height, length = cqt.shape
    new_length = length // mean_size
    new_cqt = np.zeros((height, new_length), dtype=np.float64)
    for i in range(new_length):
        new_cqt[:, i] = cqt[:, i*mean_size:(i+1)*mean_size].mean(axis=1)
    return new_cqt

## Dataset Processing

In [26]:
os.makedirs(CQT_DIR, exist_ok=True)

file_list = []
song_dirs = sorted([d for d in os.listdir(DATA_DIR)
                    if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f"Found {len(song_dirs)} song directories")

for song_id, song_dir in enumerate(song_dirs):
    song_path = os.path.join(DATA_DIR, song_dir)
    mp3_files = glob.glob(os.path.join(song_path, '*.mp3'))

    for cover_id, mp3_path in enumerate(sorted(mp3_files)):
        try:
            cqt_features = compute_cqt_features(mp3_path)
            filename = f"{song_id}_{cover_id}.npy"
            filepath = os.path.join(CQT_DIR, filename)
            np.save(filepath, cqt_features)
            file_list.append((filepath, song_id))
        except Exception as e:
            print(f"ERROR processing {mp3_path}: {e}")

NUM_CLASSES = len(song_dirs)
print(f"Processed {len(file_list)} files, {NUM_CLASSES} classes")

Found 3 song directories
Processed 27 files, 3 classes


## Dataset Class

In [27]:
class CQTDataset(Dataset):
    def __init__(self, file_list, out_length=200, is_training=True):
        self.file_list = file_list
        self.out_length = out_length
        self.is_training = is_training

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        filepath, label = self.file_list[idx]
        cqt = np.load(filepath).T  # (T, 84)

        mean, std = cqt.mean(), cqt.std()
        if std > 0:
            cqt = (cqt - mean) / std

        T = cqt.shape[0]
        if T >= self.out_length:
            if self.is_training:
                start = random.randint(0, T - self.out_length)
            else:
                start = (T - self.out_length) // 2
            cqt = cqt[start:start + self.out_length, :]
        else:
            padding = np.zeros((self.out_length - T, 84))
            cqt = np.concatenate([cqt, padding], axis=0)

        cqt = cqt.T[np.newaxis, :, :]  # (1, 84, out_length)
        return torch.FloatTensor(cqt), label

## Train/Test Split

In [28]:
train_files = []
test_files = []

class_files = defaultdict(list)
for filepath, label in file_list:
    class_files[label].append((filepath, label))

for label in sorted(class_files.keys()):
    files = class_files[label]
    if len(files) > 1:
        train_files.extend(files[:-1])
        test_files.extend(files[-1:])
    else:
        train_files.extend(files)

print(f"Train: {len(train_files)}, Test: {len(test_files)}")

train_dataset = CQTDataset(train_files, out_length=OUT_LENGTH, is_training=True)
test_dataset = CQTDataset(test_files, out_length=OUT_LENGTH, is_training=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2)

Train: 24, Test: 3


## CQTNet Model

In [29]:
class CQTNet(nn.Module):
    def __init__(self, num_classes=245):
        super(CQTNet, self).__init__()

        self.features = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(1, 32, kernel_size=(12, 3), dilation=(1, 1),
                                padding=(6, 0), bias=False)),
            ('norm0', nn.BatchNorm2d(32)),
            ('relu0', nn.ReLU(inplace=True)),

            ('conv1', nn.Conv2d(32, 64, kernel_size=(13, 3), dilation=(1, 2),
                                bias=False)),
            ('norm1', nn.BatchNorm2d(64)),
            ('relu1', nn.ReLU(inplace=True)),
            ('pool1', nn.MaxPool2d((1, 2), stride=(1, 2), padding=(0, 1))),

            ('conv2', nn.Conv2d(64, 64, kernel_size=(13, 3), dilation=(1, 1),
                                bias=False)),
            ('norm2', nn.BatchNorm2d(64)),
            ('relu2', nn.ReLU(inplace=True)),

            ('conv3', nn.Conv2d(64, 64, kernel_size=(3, 3), dilation=(1, 2),
                                bias=False)),
            ('norm3', nn.BatchNorm2d(64)),
            ('relu3', nn.ReLU(inplace=True)),
            ('pool3', nn.MaxPool2d((1, 2), stride=(1, 2), padding=(0, 1))),

            ('conv4', nn.Conv2d(64, 128, kernel_size=(3, 3), dilation=(1, 1),
                                bias=False)),
            ('norm4', nn.BatchNorm2d(128)),
            ('relu4', nn.ReLU(inplace=True)),

            ('conv5', nn.Conv2d(128, 128, kernel_size=(3, 3), dilation=(1, 2),
                                bias=False)),
            ('norm5', nn.BatchNorm2d(128)),
            ('relu5', nn.ReLU(inplace=True)),
            ('pool5', nn.MaxPool2d((1, 2), stride=(1, 2), padding=(0, 1))),

            ('conv6', nn.Conv2d(128, 256, kernel_size=(3, 3), dilation=(1, 1),
                                bias=False)),
            ('norm6', nn.BatchNorm2d(256)),
            ('relu6', nn.ReLU(inplace=True)),

            ('conv7', nn.Conv2d(256, 256, kernel_size=(3, 3), dilation=(1, 2),
                                bias=False)),
            ('norm7', nn.BatchNorm2d(256)),
            ('relu7', nn.ReLU(inplace=True)),
            ('pool7', nn.MaxPool2d((1, 2), stride=(1, 2), padding=(0, 1))),

            ('conv8', nn.Conv2d(256, 512, kernel_size=(3, 3), dilation=(1, 1),
                                bias=False)),
            ('norm8', nn.BatchNorm2d(512)),
            ('relu8', nn.ReLU(inplace=True)),

            ('conv9', nn.Conv2d(512, 512, kernel_size=(3, 3), dilation=(1, 2),
                                bias=False)),
            ('norm9', nn.BatchNorm2d(512)),
            ('relu9', nn.ReLU(inplace=True)),
        ]))

        self.pool = nn.AdaptiveMaxPool2d((1, 1))
        self.fc0 = nn.Linear(512, 300)
        self.fc1 = nn.Linear(300, num_classes)

    def forward(self, x):
        N = x.size(0)
        x = self.features(x)   # (N, 512, H, W)
        x = self.pool(x)       # (N, 512, 1, 1)
        x = x.view(N, -1)     # (N, 512)
        embedding = self.fc0(x)  # (N, 300)
        logits = self.fc1(embedding)  # (N, num_classes)
        return logits, embedding

## Training

In [30]:
model = CQTNet(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  factor=0.5, patience=5)

# adjust batch size if dataset is smaller than BATCH_SIZE
actual_batch_size = min(BATCH_SIZE, len(train_files))
train_loader = DataLoader(train_dataset, batch_size=actual_batch_size, shuffle=True,
                          num_workers=2, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=actual_batch_size, shuffle=False,
                         num_workers=2)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0

    for data, labels in train_loader:
        data, labels = data.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(data)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        _, predicted = torch.max(logits, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = epoch_loss / len(train_loader)
    accuracy = 100.0 * correct / total
    scheduler.step(avg_loss)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  Loss: {avg_loss:.4f}  Acc: {accuracy:.1f}%")


Epoch 1/100  Loss: 1.0743  Acc: 29.2%
Epoch 2/100  Loss: 3.8634  Acc: 50.0%
Epoch 3/100  Loss: 3.4675  Acc: 20.8%
Epoch 4/100  Loss: 1.9732  Acc: 54.2%
Epoch 5/100  Loss: 1.5203  Acc: 50.0%
Epoch 6/100  Loss: 1.2876  Acc: 50.0%
Epoch 7/100  Loss: 0.9229  Acc: 45.8%
Epoch 8/100  Loss: 1.3806  Acc: 29.2%
Epoch 9/100  Loss: 1.0922  Acc: 33.3%
Epoch 10/100  Loss: 0.7787  Acc: 45.8%
Epoch 11/100  Loss: 0.8863  Acc: 50.0%
Epoch 12/100  Loss: 0.8859  Acc: 41.7%
Epoch 13/100  Loss: 1.0513  Acc: 62.5%
Epoch 14/100  Loss: 1.0372  Acc: 70.8%
Epoch 15/100  Loss: 1.0363  Acc: 70.8%
Epoch 16/100  Loss: 0.8078  Acc: 70.8%
Epoch 17/100  Loss: 0.8679  Acc: 62.5%
Epoch 18/100  Loss: 0.6708  Acc: 75.0%
Epoch 19/100  Loss: 0.6810  Acc: 75.0%
Epoch 20/100  Loss: 0.7617  Acc: 62.5%
Epoch 21/100  Loss: 0.7253  Acc: 62.5%
Epoch 22/100  Loss: 0.7330  Acc: 66.7%
Epoch 23/100  Loss: 0.6985  Acc: 75.0%
Epoch 24/100  Loss: 0.6510  Acc: 66.7%
Epoch 25/100  Loss: 0.7482  Acc: 62.5%
Epoch 26/100  Loss: 0.6519  Acc: 7

In [31]:
model.eval()
with torch.no_grad():
    for data, labels in test_loader:
        data, labels = data.to(device), labels.to(device)
        logits, _ = model(data)
        _, predicted = torch.max(logits, dim=1)

        for i in range(len(labels)):
            status = "CORRECT" if predicted[i] == labels[i] else "WRONG"
            print(f"Sample {i} | True: Song {labels[i].item()} | Predicted: Song {predicted[i].item()} | {status}")

# overall test accuracy
all_correct = 0
all_total = 0
with torch.no_grad():
    for data, labels in test_loader:
        data, labels = data.to(device), labels.to(device)
        logits, _ = model(data)
        _, predicted = torch.max(logits, dim=1)
        all_total += labels.size(0)
        all_correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {all_correct}/{all_total} = {100.0 * all_correct / all_total:.1f}%")


Sample 0 | True: Song 0 | Predicted: Song 0 | CORRECT
Sample 1 | True: Song 1 | Predicted: Song 1 | CORRECT
Sample 2 | True: Song 2 | Predicted: Song 2 | CORRECT
Test Accuracy: 3/3 = 100.0%


## Evaluation

In [32]:
def extract_embeddings(model, data_loader, device):
    model.eval()
    all_embeddings = []
    all_labels = []
    with torch.no_grad():
        for data, labels in data_loader:
            data = data.to(device)
            _, embedding = model(data)
            all_embeddings.append(embedding.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_embeddings), np.concatenate(all_labels)


def compute_map(similarity_matrix, labels):
    N = len(labels)
    aps = []
    for i in range(N):
        sims = similarity_matrix[i].copy()
        sims[i] = -1.0
        relevant = (labels == labels[i])
        relevant[i] = False
        num_relevant = relevant.sum()
        if num_relevant == 0:
            continue
        ranked = np.argsort(-sims)
        hits = 0
        sum_prec = 0.0
        for rank, idx in enumerate(ranked):
            if idx == i:
                continue
            if relevant[idx]:
                hits += 1
                sum_prec += hits / (rank + 1)
            if hits == num_relevant:
                break
        aps.append(sum_prec / num_relevant)
    return np.mean(aps)

In [33]:
all_dataset = CQTDataset(file_list, out_length=OUT_LENGTH, is_training=False)
all_loader = DataLoader(all_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

embeddings, labels = extract_embeddings(model, all_loader, device)

# L2 normalize
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings / np.maximum(norms, 1e-8)

similarity_matrix = embeddings_norm @ embeddings_norm.T
mean_ap = compute_map(similarity_matrix, labels)
print(f"Mean Average Precision (MAP): {mean_ap:.4f}")

Mean Average Precision (MAP): 0.9863


## Save Model

In [34]:
MODEL_PATH = '/content/drive/MyDrive/cqtnet_indian_music.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'num_classes': NUM_CLASSES,
    'epoch': NUM_EPOCHS,
}, MODEL_PATH)

print(f"Model saved to {MODEL_PATH}")

Model saved to /content/drive/MyDrive/cqtnet_indian_music.pth
